In [25]:
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics.pairwise import cosine_distances

nltk.download("all")

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nl

True

In [26]:
data = pd.read_csv("data/additional_features.csv")
# remove songs with missing lyrics and reset the index
data = data.dropna(subset=['lyrics']).reset_index(drop=True)
data

,id,Rank,Title,Primary Artists,Featuring Artists,Date,Danceability,Energy,Loudness,Speechiness,...,lyrics,detected_languages,num_languages,language_confidences,artist_id,explicit,track_popularity,artist_followers,artist_genres,artist_popularity
0,47EiUVwUp4C9fGccaPuUCS,1,dákiti,"bad bunny, jhayco",NaN,26/11/2020,0.731,0.573,-10059.00,0.054,...,"Baby, ya yo me enteré, se nota cuando me ve'\n...",SPANISH,1,SPANISH: 1.0,4q3ewBCX7sLwd24euuV69X,True,78,102185553.0,"reggaeton, trap latino, urbano latino, latin",95.0
1,7igeByaBM0MgGsgXtNxDJ7,1,positions,ariana grande,NaN,29/10/2020,0.736,0.802,-4759.00,0.086,...,Heaven sent you to me\nI'm just hopin’ I don't...,ENGLISH,1,ENGLISH: 1.0,66CXWjxzNUsdJxJ2JdwvnR,True,0,107261237.0,pop,90.0
2,3qQbCzHBycnDpGskqOWY0E,1,ella baila sola,"eslabon armado, peso pluma",NaN,29/05/2023,0.668,0.758,-5176.00,0.033,...,"Compa, ¿qué le parece esa morra?\nLa que anda ...",SPANISH,1,SPANISH: 1.0,0XeEobZplHxzM9QzFQWLiR,False,73,6434994.0,"corridos tumbados, sad sierreño, corrido, sier...",75.0
3,0Yde1VrWbGF2Tus2AQhMuT,1,circles,post malone,NaN,06/09/2019,0.703,0.756,-3407.00,0.038,...,"Oh, oh, oh\nOh, oh, oh\nOh, oh, oh, oh, oh\n...",ENGLISH,1,ENGLISH: 1.0,246dkjvS1zLTtiykXe5h60,False,0,47686394.0,NaN,85.0
4,0TK2YIli7K1leLovkQiNik,1,señorita,"camila cabello, shawn mendes",NaN,25/07/2019,0.759,0.540,-6039.00,0.029,...,I love it when you call me señorita\nI wish I ...,ENGLISH,1,ENGLISH: 1.0,7n2wHs1TKAczGzO7Dd2rGr,False,79,45665812.0,NaN,81.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8029,7wBJfHzpfI3032CSD7CE2m,200,stargazing,travis scott,NaN,14/05/2020,0.487,0.789,-3.69,0.044,...,"Ah\n\nRollin', rollin', rollin', got me starga...",ENGLISH,1,ENGLISH: 1.0,0Y5tJX1MQlPlqiwlOH1tJY,True,73,41006554.0,rap,89.0
8030,7yotKA30dwTKNEGomV9ZsI,200,kod,j. cole,NaN,28/06/2018,0.823,0.783,-6274.00,0.154,...,"Go, go, go\nGo, go, go\nGo, go, go\nGo, go, go...",ENGLISH,1,ENGLISH: 1.0,6l3HvQ5sa6mXTsMTB19rO5,True,60,27037715.0,rap,82.0
8031,5FOCkGt7bp22nBVfQ4WSIT,200,indecente,anitta,NaN,19/04/2018,0.874,0.602,-5118.00,0.127,...,"Una mirada te delata, un perfume que te atrapa...",SPANISH,1,SPANISH: 1.0,7FNnA9vBm6EKceENgCGRMb,False,48,13245754.0,"brazilian pop, funk pop, brazilian funk, funk ...",78.0
8032,7zQirOExB0VR8yWUOqYeio,200,dolla sign slime (feat. megan thee stallion),lil nas x,megan thee stallion,27/09/2021,0.923,0.533,-6016.00,0.262,...,Day-Day-Daytrip took it to ten (\nHey\n)\n\nY'...,ENGLISH,1,ENGLISH: 1.0,7jVv8c5Fj3E9VhNjxT4snq,True,50,12311640.0,NaN,71.0


In [27]:
# Build semantic embeddings for lyrics, find similar songs, and train a simple prediction model.
# Uses sentence-transformers if available, otherwise falls back to TF-IDF.
lyrics = data['lyrics'].fillna("").astype(str).tolist()

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
embeddings_sparse = vectorizer.fit_transform(lyrics)
embeddings = embeddings_sparse  # keep sparse matrix
embeddings_is_sparse = True

nn = NearestNeighbors(n_neighbors=6, metric='cosine', n_jobs=-1)
nn.fit(embeddings)

def get_similar(song_idx: int, k: int = 5):
    """
    Return k most similar songs (index, distance, title, primary artists).
    Excludes the query itself from the results.
    """
    query = embeddings[song_idx] if embeddings_is_sparse else embeddings[song_idx]
    distances, indices = nn.kneighbors([query] if not embeddings_is_sparse else query, n_neighbors=k+1)
    distances = distances[0]
    indices = indices[0]
    results = []
    for dist, idx in zip(distances, indices):
        if idx == song_idx:
            continue
        row = data.iloc[idx]
        results.append({'idx': int(idx), 'distance': float(dist), 'title': row['Title'], 'artists': row['Primary Artists']})
        if len(results) >= k:
            break 
    return results

In [28]:
def get_songs_by_artist(artist: str) -> pd.DataFrame:
    a = artist.strip().lower()
    prim = data['Primary Artists'].fillna('').astype(str)
    feat = data['Featuring Artists'].fillna('').astype(str)

    pattern = re.escape(a)
    mask_prim = prim.str.lower().str.contains(pattern, na=False)
    mask_feat = feat.str.lower().str.contains(pattern, na=False)

    return data[mask_prim | mask_feat].copy()

def get_lyrics_by_title(title: str, exact: bool = True, first_only: bool = True):
    t = title.strip().lower()
    titles = data['Title'].fillna('').astype(str).str.lower()
    if exact:
        mask = titles == t
    else:
        mask = titles.str.contains(re.escape(t), na=False)
    matches = data[mask]

    if matches.empty:
        return None if first_only else matches
    if first_only:
        row = matches.iloc[0]
        return row['lyrics']
    
    return matches[['lyrics']].copy()

In [29]:
# Example: find a song by title and show 5 similar songs
title_query = "unsainted"
matches = data[data['Title'].fillna('').str.lower().str.contains(title_query.lower())]

if matches.empty:
    print(f"No songs found matching title '{title_query}'")
else:
    example_idx = int(matches.index[0])
    row = data.loc[example_idx]
    print("Query:", example_idx, row['Title'], "|", row.get('Primary Artists', ''))
    for r in get_similar(example_idx, k=10):
        print(f"\t{r['idx']}: {r['title']} | {r['artists']} (distance={r['distance']:.4f})")

Query: 1256 unsainted | slipknot
	3827: unsainted | slipknot (distance=0.0000)
	7672: in the stars | benson boone (distance=0.8106)
	6654: caroli | harry styles (distance=0.8245)
	4849: believe | eminem (distance=0.8281)
	4338: never love again | eminem (distance=0.8407)
	2637: favorite bitch (feat. ty dolla $ign) | eminem (distance=0.8427)
	7603: arose | eminem (distance=0.8457)
	7999: the ringer | eminem (distance=0.8460)
	2560: wilshire | the creator, tyler (distance=0.8465)
	6389: killshot | eminem (distance=0.8484)


In [30]:
features = ["Danceability", "Energy", "Loudness", "Speechiness", "Acousticness", "Instrumentalness", "Valence"]

In [31]:
# Install sentence-transformers in the notebook environment if it's missing.
%pip install -q sentence-transformers

from pathlib import Path
from sentence_transformers import SentenceTransformer

# Compute/Load semantic embeddings (sentence-transformers) and rank by semantic + feature similarity.
SEM_EMB_PATH = Path("data/semantic_embeddings.npy")
lyrics_list = data['lyrics'].fillna("").astype(str).tolist()

def ensure_semantic_embeddings(model_name: str = "all-MiniLM-L6-v2", force_recompute: bool = False) -> np.ndarray:
    global embeddings_dense, embeddings_is_sparse

    # If already in memory and not forced, return it
    if 'embeddings_dense' in globals() and not force_recompute:
        return embeddings_dense

    # Load from disk if available
    if SEM_EMB_PATH.exists() and not force_recompute:
        embeddings_dense = np.load(SEM_EMB_PATH)
        embeddings_is_sparse = False
        return embeddings_dense

    model = SentenceTransformer(model_name)
    # encode with batched inference; returns numpy array
    embeddings_dense = model.encode(lyrics_list, show_progress_bar=True, convert_to_numpy=True)
    np.save(SEM_EMB_PATH, embeddings_dense)
    embeddings_is_sparse = False
    return embeddings_dense

def get_ranked_recommendations(song_list: pd.DataFrame, n_recs: int = 10, semantic=True, semantic_weight: float = 0.7) -> pd.DataFrame:
    if song_list.empty:
        return pd.DataFrame(columns=['Title', 'Primary Artists', 'Featuring Artists', 'Feature_Similarity', 'Semantic_Similarity', 'Combined_Score'])

    feat_cols = features

    # Impute & scale audio features on full dataset
    impute_vals = data[feat_cols].median()
    X_full_imputed = data[feat_cols].fillna(impute_vals)
    scaler = StandardScaler().fit(X_full_imputed)
    X_scaled = pd.DataFrame(scaler.transform(X_full_imputed), index=X_full_imputed.index, columns=feat_cols)

    liked_idx = song_list.index.to_numpy()
    liked_feat = X_scaled.loc[liked_idx].values
    feat_dists = cosine_distances(liked_feat, X_scaled.values)
    feat_sims_matrix = 1 - feat_dists
    feat_sim_agg = np.median(feat_sims_matrix, axis=0)

    # Semantic similarities
    semantic_sim_agg = None

    if semantic:
        try:
            emb_dense = ensure_semantic_embeddings()
            liked_emb = emb_dense[liked_idx]
            sem_dists = cosine_distances(liked_emb, emb_dense)
            sem_sims_matrix = 1 - sem_dists
            semantic_sim_agg = np.median(sem_sims_matrix, axis=0)
        except Exception as e:
            print("Semantic embeddings unavailable; falling back to existing embeddings for lyric similarity. Error:", str(e))

    # If semantic not produced, fall back to existing `embeddings` (TF-IDF)
    if semantic_sim_agg is None or not semantic:
        if 'embeddings' in globals() and embeddings is not None:
            liked_emb = embeddings[liked_idx]
            sem_dists = cosine_distances(liked_emb, embeddings)
            sem_sims_matrix = 1 - sem_dists
            semantic_sim_agg = np.median(sem_sims_matrix, axis=0)
        else:
            semantic_sim_agg = np.zeros(len(data))

    # Combined score: semantic_weight governs semantics vs features
    combined = semantic_weight * semantic_sim_agg + (1.0 - semantic_weight) * feat_sim_agg

    scores = pd.DataFrame({
        'Feature Similarity': feat_sim_agg,
        'Lyric Similarity': semantic_sim_agg,
        'Combined Score': combined
    }, index=X_scaled.index)

    # Exclude liked songs robustly (works for any index type) and pick top n_recs
    mask = ~scores.index.isin(liked_idx)
    top = scores.loc[mask].sort_values('Combined Score', ascending=False).head(n_recs)

    recs = data.loc[top.index, ['Rank', 'Title', 'Primary Artists', 'Featuring Artists'] + feat_cols].copy()
    recs['Feature Similarity'] = top['Feature Similarity'].values
    recs['Lyric Similarity'] = top['Lyric Similarity'].values
    recs['Combined Score'] = top['Combined Score'].values

    recs = recs.reset_index().drop(columns=['index'])
    return recs

Note: you may need to restart the kernel to use updated packages.


In [32]:
artist = "drake"
artist_songs = get_songs_by_artist(artist)
liked_songs = artist_songs.head(1) if len(artist_songs) > 1 else artist_songs

ranked_recs_tfidf = get_ranked_recommendations(liked_songs, n_recs=10, semantic=False, semantic_weight=0.7)
ranked_recs_semantic = get_ranked_recommendations(liked_songs, n_recs=10, semantic=True, semantic_weight=0.7)

print(f"Artist: {artist}")
display(liked_songs["Title"])

print("Semantic + Feature Recommendations:")
display(ranked_recs_semantic)

print("TF-IDF + Feature Recomendation")
display(ranked_recs_tfidf)

Artist: drake


14    in my feelings
Name: Title, dtype: object

Semantic + Feature Recommendations:


,Rank,Title,Primary Artists,Featuring Artists,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Feature Similarity,Lyric Similarity,Combined Score
0,191,in my feelings,drake,NaN,0.835,0.626,-5833.0,0.125,0.059,0.0,0.350,0.574377,1.000000,0.872313
1,184,sin (feat. jaden smith),young thug,jaden smith,0.889,0.385,-11767.0,0.165,0.005,0.0,0.235,0.950040,0.638506,0.731966
2,200,in my head,lil tjay,NaN,0.684,0.545,-6885.0,0.121,0.159,0.0,0.407,0.920305,0.649714,0.730892
3,177,ztfo,big sean,NaN,0.720,0.542,-7583.0,0.048,0.051,0.0,0.374,0.830682,0.679302,0.724716
4,190,incredible,future,NaN,0.790,0.438,-6462.0,0.133,0.063,0.0,0.247,0.825821,0.668945,0.716008
5,102,perplexing pegasus,rae sremmurd,NaN,0.898,0.447,-8849.0,0.077,0.001,0.0,0.308,0.865657,0.647107,0.712672
6,196,perplexing pegasus,rae sremmurd,NaN,0.899,0.448,-8789.0,0.074,0.001,0.0,0.302,0.861707,0.647107,0.711487
7,177,afterglow,taylor swift,NaN,0.756,0.449,-8746.0,0.034,0.130,0.0,0.399,0.818617,0.664955,0.711053
8,157,"12.38 (feat. 21 savage, ink & kadhja bonet)",childish gambino,"21 savage, ink, kadhja bonet",0.691,0.524,-9718.0,0.160,0.197,0.0,0.410,0.863496,0.642415,0.708739
9,198,falling,trevor daniel,NaN,0.784,0.430,-8756.0,0.036,0.123,0.0,0.236,0.850480,0.646497,0.707692


TF-IDF + Feature Recomendation


,Rank,Title,Primary Artists,Featuring Artists,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Feature Similarity,Lyric Similarity,Combined Score
0,191,in my feelings,drake,NaN,0.835,0.626,-5833.0,0.125,0.059,0.0,0.350,0.574377,1.000000,0.872313
1,200,you got it,vedo,NaN,0.762,0.433,-8937.0,0.187,0.143,0.0,0.394,0.894402,0.152291,0.374924
2,172,pelle coat,lil durk,NaN,0.747,0.565,-8859.0,0.096,0.004,0.0,0.395,0.929012,0.127115,0.367684
3,157,"12.38 (feat. 21 savage, ink & kadhja bonet)",childish gambino,"21 savage, ink, kadhja bonet",0.691,0.524,-9718.0,0.160,0.197,0.0,0.410,0.863496,0.142754,0.358976
4,200,in my head,lil tjay,NaN,0.684,0.545,-6885.0,0.121,0.159,0.0,0.407,0.920305,0.115768,0.357129
5,182,bëttr 0ff,yeat,NaN,0.731,0.532,-9815.0,0.074,0.002,0.0,0.370,0.933177,0.107010,0.354860
6,184,sin (feat. jaden smith),young thug,jaden smith,0.889,0.385,-11767.0,0.165,0.005,0.0,0.235,0.950040,0.099567,0.354709
7,200,monëy so big,yeat,NaN,0.693,0.575,-7633.0,0.091,0.121,0.0,0.353,0.901935,0.119779,0.354426
8,192,mean it,"lany, lauv",NaN,0.746,0.450,-8543.0,0.087,0.041,0.0,0.336,0.948875,0.097482,0.352900
9,195,mean it,"lany, lauv",NaN,0.746,0.450,-8543.0,0.087,0.041,0.0,0.336,0.948875,0.097482,0.352900
